# How does a computer find cells in a photo? 🖼️

Every number in the last notebook started as a **picture**. Somebody had to
turn the picture into a table — to draw an outline around each cell and say
"this is cell #1, this is cell #2."

That's called **segmentation**. Let's try it.

In [ ]:
# --- setup: run me first! ---
import pandas as pd, numpy as np, matplotlib.pyplot as plt

DATA = "https://raw.githubusercontent.com/emcramer/psi-class-2026/main/data/processed"

plt.rcParams.update({"figure.figsize": (8, 5.5), "font.size": 13,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e1e0d9",
                     "figure.facecolor": "white", "axes.facecolor": "white"})
BLUE, ORANGE, GREEN = "#2a78d6", "#eb6834", "#1baf7a"
print("ready")

In [ ]:
import urllib.request
urllib.request.urlretrieve(f"{DATA}/seg_demo.npz", "seg_demo.npz")
d = np.load("seg_demo.npz")

plt.figure(figsize=(7, 7))
plt.imshow(d["easy"], cmap="gray")
plt.title("A microscope image. How many cells?"); plt.axis("off")
plt.show()

You can see the blobs. Counting them by hand would take a while — and there are
**200,000** of these in the full dataset.

The classic recipe is three steps:

1. **Threshold** — anything brighter than X is a cell, anything darker is background
2. **Distance transform** — find the center of each blob
3. **Watershed** — flood outward from each center until the floods meet

In [ ]:
from scipy import ndimage as ndi
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
from skimage.filters import threshold_otsu, gaussian

def find_cells(image, min_distance=4):
    smooth = gaussian(image, 1.0)
    is_cell = smooth > threshold_otsu(smooth)          # 1. threshold
    distance = ndi.distance_transform_edt(is_cell)     # 2. distance to edge
    peaks = peak_local_max(distance, min_distance=min_distance, labels=is_cell)
    seeds = np.zeros(distance.shape, bool); seeds[tuple(peaks.T)] = True
    return watershed(-distance, ndi.label(seeds)[0], mask=is_cell)  # 3. flood

found = find_cells(d["easy"])
print("cells found:", found.max())
print("cells actually there:", len(np.unique(d["nuclei"])) - 1)

In [ ]:
def show(image, segmentation, title):
    rng = np.random.default_rng(1)
    shuffle = np.concatenate([[0], rng.permutation(np.arange(1, segmentation.max() + 1))])
    fig, ax = plt.subplots(1, 2, figsize=(12, 6))
    ax[0].imshow(image, cmap="gray"); ax[0].set_title("the picture")
    ax[1].imshow(np.where(segmentation > 0, shuffle[segmentation], np.nan),
                 cmap="tab20", interpolation="nearest")
    ax[1].set_title("cells the computer found")
    for a in ax: a.axis("off"); a.grid(False)
    fig.suptitle(title, fontsize=15)
    plt.show()

show(d["easy"], found, "Nice, well-separated cells")

That worked well! Three lines of math, no "AI" at all.

### Now the catch.

Real tumor tissue isn't tidy. Cells are jammed against each other with no gaps.
Here's what an actual sample looks like — **same code, harder picture:**

In [ ]:
found_hard = find_cells(d["hard"], min_distance=5)
show(d["hard"], found_hard, "Real, densely packed tissue")

def outline_accuracy(truth, guess):
    scores = []
    for label in np.unique(truth):
        if label == 0: continue
        mask = truth == label
        overlap = guess[mask][guess[mask] > 0]
        if len(overlap) == 0: scores.append(0); continue
        counts = np.bincount(overlap); best = counts.argmax()
        scores.append(counts[best] / (mask | (guess == best)).sum())
    return np.mean(scores)

print(f"easy picture: outlines {outline_accuracy(d['nuclei'], found):.0%} correct")
print(f"real tissue:  outlines {outline_accuracy(d['truth'], found_hard):.0%} correct")

### It falls apart.

Look at the big merged blobs — the algorithm glued neighboring cells together.
It got roughly the right *count*, but the *shapes* are wrong, and every protein
measurement you make from a wrong shape is also wrong.

**This is exactly why the scientists who made our dataset used a neural
network instead.** They trained one on thousands of hand-drawn cell outlines
until it learned what a cell boundary looks like — the same idea as your phone
recognizing a face.

The table you used in notebook 1? A neural network drew every one of those
200,000 outlines.

---
### 🚀 If you have time

- Change `min_distance` and watch cells split apart or merge together.
- Where else does "label every pixel" matter? (Self-driving cars. Tumor
  outlines for radiation therapy. Satellite maps of deforestation.)